In [1]:
# SYSTEM
import os

# DOCX Processing
from docx import Document

# LangChain & FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings

# Sentence Transformers for embedding models
from sentence_transformers import SentenceTransformer

# Hugging Face login for gated models
from huggingface_hub import login

# Transformers & LangChain LLM
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain.llms import HuggingFacePipeline

# LangChain retrieval Q&A
from langchain.chains import RetrievalQA


In [4]:
from huggingface_hub import login
login()


In [5]:
def load_all_docx(folder_path):
    all_texts = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".docx"):
            file_path = os.path.join(folder_path, filename)
            doc = Document(file_path)
            text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
            all_texts.append(text)
    return all_texts

In [6]:
# Example Windows path
folder_path = r"C:\USDA\dataset"
texts = load_all_docx(folder_path)


In [7]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = splitter.create_documents(texts)

print(f"✅ Loaded and split {len(documents)} document chunks.")


✅ Loaded and split 73 document chunks.


In [8]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(documents, embedding_model)
db.save_local("faiss_hf_index")

print("✅ FAISS index created and saved locally.")


C:\Users\axi034\AppData\Local\Temp\ipykernel_1372\7504430.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\axi034\AppData\Local\anaconda3\envs\rag-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\axi034\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ FAISS index created and saved locally.


In [9]:
model_name = "mistralai/Mistral-7B-Instruct-v0.1"  # Change if using another

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto"
)

llm = HuggingFacePipeline(pipeline=pipe)

print("✅ Model loaded.")


tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

C:\Users\axi034\AppData\Local\anaconda3\envs\rag-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\axi034\.cache\huggingface\hub\models--mistralai--Mistral-7B-Instruct-v0.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.
Device set to use cuda:0


✅ Model loaded.


C:\Users\axi034\AppData\Local\Temp\ipykernel_1372\2887606557.py:13: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [10]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 4})

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("✅ RAG pipeline initialized.")


✅ RAG pipeline initialized.


In [ ]:
while True:
    query = input("\n📝 Enter your question (or type 'exit' to quit): ")
    if query.lower() == 'exit':
        print("👋 Session ended.")
        break

    result = rag_chain({"query": query})
    print("\n🎯 Answer:\n", result["result"])

    again = input("\n❓ Do you want to ask another question? (yes/no): ")
    if again.lower() not in ['yes', 'y']:
        print("👋 Session ended.")
        break



📝 Enter your question (or type 'exit' to quit):  Explain how the life cycle of the Asian Citrus Psyllid influences management strategies?


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



🎯 Answer:
 Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

The rapid life cycle of the Asian citrus psyllid, characterized by its quick progression from egg to adult in just over two weeks and the potential for numerous generations annually, presents a significant and dynamic challenge for effective pest management. This accelerated reproductive capacity means that psyllid populations can rebound with extreme rapidity following any intervention, necessitating continuous and precisely timed management efforts. The dependence of the nymphal stages on the new flush growth of host plants further intertwines psyllid population dynamics with the physiological growth cycles of citrus trees. This linkage implies that control strategies, such as the application of pesticides, must be carefully synchronized with these plant growth phases, which can vary regionally and seasonally. 